In [1]:
#!/usr/bin/env python
# coding: utf-8

# =============================================================================
# ADNI LOCAL PIPELINE - COMPLETE CONSOLIDATED NOTEBOOK
# Replicates full Google Colab workflow with all fixes for local Windows
# =============================================================================

# =============================================================================
# CELL 1: ENVIRONMENT SETUP & IMPORTS
# =============================================================================

import os
import sys
import shutil
import subprocess
import glob
from collections import Counter
import numpy as np
import pandas as pd
import nibabel as nib
import SimpleITK as sitk
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.utils import class_weight
import joblib
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("ADNI LOCAL PIPELINE - WINDOWS")
print("="*60)
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

ADNI LOCAL PIPELINE - WINDOWS
Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
PyTorch: 2.7.1+cpu
CUDA available: False


In [2]:


# =============================================================================
# CELL 2: CONFIGURATION - EDIT THESE PATHS
# =============================================================================

# --- INPUT PATHS ---
# Your NIfTI files (AD, MCI, CN folders inside)
NIFTI_BASE = r"E:\ADNI_NIfTI"

# ADNIMERGE.csv file path
ADNIMERGE_PATH = r"E:\ADNIMERGE.csv"

# FastSurfer installation path (from git clone)
FASTSURFER_ROOT = r"E:\ADNI_Local\FastSurfer"

# --- OUTPUT PATHS ---
OUTPUT_ROOT = r"E:\ADNI_Local\outputs"
SAVE_DIR = os.path.join(OUTPUT_ROOT, "processed_data")
FS_OUTPUT = os.path.join(OUTPUT_ROOT, "fastsurfer_results")
HIPPO_CROP_DIR = os.path.join(OUTPUT_ROOT, "hippo_crops")

# Create directories
for d in [OUTPUT_ROOT, SAVE_DIR, FS_OUTPUT, HIPPO_CROP_DIR]:
    os.makedirs(d, exist_ok=True)

# --- CONSTANTS ---
CATEGORIES = ['AD', 'MCI', 'CN']
LABEL_MAP = {'CN': 0, 'MCI': 1, 'AD': 2}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

# FreeSurfer/FastSurfer label IDs for regions
REGION_LABELS = {
    'left_hippo': 17, 'right_hippo': 53,
    'left_ventricle': 4, 'right_ventricle': 43,
    'left_entorhinal': 1006, 'right_entorhinal': 2006,
    'left_fusiform': 1007, 'right_fusiform': 2007,
    'left_midtemp': 1015, 'right_midtemp': 2015
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {DEVICE}")
print(f"NIfTI base: {NIFTI_BASE}")
print(f"ADNIMERGE: {ADNIMERGE_PATH}")
print(f"Output: {OUTPUT_ROOT}")
print(f"FastSurfer: {FASTSURFER_ROOT}")


Using device: cpu
NIfTI base: E:\ADNI_NIfTI
ADNIMERGE: E:\ADNIMERGE.csv
Output: E:\ADNI_Local\outputs
FastSurfer: E:\ADNI_Local\FastSurfer


In [3]:

# =============================================================================
# CELL 3: IDENTIFY CONVERTER PATIENTS (from ADNIMERGE.csv)
# =============================================================================

def identify_converters(adnimerge_path):
    """
    Identify subjects who converted from CN/MCI to AD.
    These are excluded from training to prevent data leakage.
    """
    print("\n" + "="*60)
    print("STEP 1: Identifying Converter Patients")
    print("="*60)

    if not os.path.exists(adnimerge_path):
        print(f"WARNING: {adnimerge_path} not found. No converters excluded.")
        return set()

    merge_df = pd.read_csv(adnimerge_path, low_memory=False)
    subjects = merge_df.sort_values(['PTID', 'VISCODE'])

    # Check available diagnosis columns
    dx_cols = [c for c in ['DX', 'DXCURREN', 'DXCHANGE'] if c in merge_df.columns]
    print(f"Available diagnosis columns: {dx_cols}")

    for col in dx_cols:
        unique_vals = merge_df[col].dropna().unique()
        print(f"  {col} unique values: {unique_vals[:10]}")

    def is_converter(group):
        # DXCHANGE codes: 4=MCI to AD, 5=CN to AD, 6=CN to MCI
        if 'DXCHANGE' in group.columns:
            if any(group['DXCHANGE'].isin([4, 5])):
                return True

        # Fallback to DX string labels
        if 'DX' in group.columns:
            labels = group['DX'].dropna().unique()
            if 'AD' in labels and len(labels) > 1:
                return True
            if 'Dementia' in labels and ('MCI' in labels or 'CN' in labels):
                return True
        return False

    converter_ids = subjects.groupby('PTID').filter(is_converter)['PTID'].unique()
    converter_set = set(converter_ids)

    print(f"\nFound {len(converter_set)} converter subjects to exclude.")

    # Save for reference
    pd.DataFrame({'PTID': list(converter_set)}).to_csv(
        os.path.join(SAVE_DIR, 'converters.csv'), index=False)

    return converter_set

converter_ids = identify_converters(ADNIMERGE_PATH)
print(f"Converter IDs sample: {list(converter_ids)[:5]}")


STEP 1: Identifying Converter Patients
Available diagnosis columns: ['DX']
  DX unique values: <StringArray>
['CN', 'Dementia', 'MCI']
Length: 3, dtype: str

Found 424 converter subjects to exclude.
Converter IDs sample: ['130_S_4542', '168_S_6467', '041_S_0549', '127_S_1032', '035_S_0204']


In [4]:

# =============================================================================
# CELL 4: BUILD CLINICAL DATAFRAME FROM NIfTI FILES
# =============================================================================

def build_clinical_df(nifti_base, converter_ids):
    """
    Scan NIfTI folders and build metadata dataframe.
    CORRECTED: Uses parts[0:3] for PTID extraction to match ADNIMERGE format.
    """
    print("\n" + "="*60)
    print("STEP 2: Building Clinical DataFrame")
    print("="*60)

    data_rows = []

    for cat in CATEGORIES:
        cat_path = os.path.join(nifti_base, cat)
        if not os.path.exists(cat_path):
            print(f"WARNING: {cat_path} not found!")
            continue

        files = [f for f in os.listdir(cat_path) if f.endswith('.nii.gz')]
        print(f"{cat}: {len(files)} NIfTI files found")

        for file in files:
            parts = file.split('_')
            # CORRECTED: PTID is first 3 parts (e.g., 002_S_0559)
            if len(parts) >= 3:
                subject_id = f"{parts[0]}_{parts[1]}_{parts[2]}"

                # Exclude converters
                if subject_id not in converter_ids:
                    data_rows.append({
                        'PTID': subject_id,
                        'mri_path': os.path.join(cat_path, file),
                        'label': cat,
                        'filename': file
                    })

    clinical_df = pd.DataFrame(data_rows)

    # Remove duplicates - keep first occurrence
    clinical_df = clinical_df.drop_duplicates(subset=['PTID'], keep='first')
    clinical_df['label_idx'] = clinical_df['label'].map(LABEL_MAP)

    print(f"\nTotal non-converter subjects: {len(clinical_df)}")
    print("\nLabel distribution:")
    print(clinical_df['label'].value_counts())

    # Save
    clinical_df.to_csv(os.path.join(SAVE_DIR, 'clinical_metadata.csv'), index=False)

    return clinical_df

clinical_df = build_clinical_df(NIFTI_BASE, converter_ids)
print("\nClinical dataframe ready!")
clinical_df.head()


STEP 2: Building Clinical DataFrame
AD: 185 NIfTI files found
MCI: 529 NIfTI files found
CN: 300 NIfTI files found

Total non-converter subjects: 196

Label distribution:
label
MCI    70
CN     67
AD     59
Name: count, dtype: int64

Clinical dataframe ready!


,PTID,mri_path,label,filename,label_idx
0,002_S_0619,E:\ADNI_NIfTI\AD\002_S_0619_2_ADNI_12M4_TS_2.n...,AD,002_S_0619_2_ADNI_12M4_TS_2.nii.gz,2
4,002_S_0816,E:\ADNI_NIfTI\AD\002_S_0816_2_ADNI_12M4_TS_2.n...,AD,002_S_0816_2_ADNI_12M4_TS_2.nii.gz,2
7,002_S_0955,E:\ADNI_NIfTI\AD\002_S_0955_2_ADNI_12M4_TS_2.n...,AD,002_S_0955_2_ADNI_12M4_TS_2.nii.gz,2
9,002_S_1018,E:\ADNI_NIfTI\AD\002_S_1018_2_ADNI_12M4_TS_2.n...,AD,002_S_1018_2_ADNI_12M4_TS_2.nii.gz,2
13,005_S_0221,E:\ADNI_NIfTI\AD\005_S_0221_2_ADNI_12M4_TS.dat...,AD,005_S_0221_2_ADNI_12M4_TS.dat_2.nii.gz,2


In [5]:
# ========== ENHANCED FEATURE EXTRACTOR ==========
# Uses more slices, adds statistical features, and 3D-aware processing

class EnhancedFeatureExtractor:
    def __init__(self):
        self.device = DEVICE
        # Use ResNet50 but extract from multiple layers
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        
        # Extract features from multiple layers for richer representation
        self.layer1 = torch.nn.Sequential(*list(model.children())[:6])   # After layer2
        self.layer2 = torch.nn.Sequential(*list(model.children())[:8])   # After layer4
        self.final = torch.nn.Sequential(*list(model.children())[:-1])   # Global pool
        
        for layer in [self.layer1, self.layer2, self.final]:
            layer.eval().to(self.device)
        
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    
    def _get_slice(self, img, axis, idx):
        """Extract a single slice and convert to RGB tensor."""
        if axis == 0: s = img[idx, :, :]
        elif axis == 1: s = img[:, idx, :]
        else: s = img[:, :, idx]
        
        # Enhance contrast for medical images
        s = (s - s.min()) / (s.max() - s.min() + 1e-5)
        s = np.clip((s - 0.3) * 2.5, 0, 1)  # Contrast enhancement
        
        rgb = np.stack([s]*3, -1)
        rgb = (rgb * 255).astype(np.uint8)
        return self.transform(Image.fromarray(rgb)).unsqueeze(0).to(self.device)
    
    def extract(self, path):
        try:
            img = sitk.GetArrayFromImage(sitk.ReadImage(path))
            img = (img - img.min()) / (img.max() - img.min() + 1e-5)
            
            h, w, d = img.shape
            all_feats = []
            
            # Sample more slices from central region
            for axis in range(3):
                dim = [h, w, d][axis]
                # Take 15 slices from central 50%
                indices = np.linspace(int(dim*0.25), int(dim*0.75), 15).astype(int)
                
                axis_feats = []
                for idx in indices:
                    tensor = self._get_slice(img, axis, idx)
                    
                    with torch.no_grad():
                        # Multi-scale features
                        f1 = self.layer1(tensor).mean([2,3]).cpu().numpy()  # 512-dim
                        f2 = self.layer2(tensor).mean([2,3]).cpu().numpy()  # 2048-dim
                        f3 = self.final(tensor).squeeze().cpu().numpy()     # 2048-dim
                        
                        combined = np.concatenate([f1.flatten(), f2.flatten(), f3.flatten()])
                        axis_feats.append(combined)
                
                # Statistical pooling across slices for this axis
                all_feats.append(np.mean(axis_feats, 0))
                all_feats.append(np.std(axis_feats, 0))
                all_feats.append(np.max(axis_feats, 0))
            
            # Final feature vector
            final = np.concatenate(all_feats)
            
            # Add volume statistics as additional features
            vol_stats = np.array([
                img.mean(), img.std(), img.max(), img.min(),
                np.percentile(img, 25), np.percentile(img, 75),
                h, w, d, h*w*d  # Spatial dimensions
            ])
            
            return np.concatenate([final, vol_stats])
            
        except Exception as e:
            print(f"Error on {os.path.basename(path)}: {e}")
            return None

print("Enhanced extractor ready!")
print("Feature dimensions: ~15,000 (multi-scale + statistical)")

Enhanced extractor ready!
Feature dimensions: ~15,000 (multi-scale + statistical)


In [6]:
# ========== EXTRACT FEATURES WITH CHECKPOINTING ==========
extractor = EnhancedFeatureExtractor()

# Check for existing features
feature_file = os.path.join(SAVE_DIR, 'X_enhanced.npy')
label_file = os.path.join(SAVE_DIR, 'y_enhanced.npy')

if os.path.exists(feature_file) and os.path.exists(label_file):
    print("Loading existing features...")
    X = np.load(feature_file)
    y = np.load(label_file)
    subjects = pd.read_csv(os.path.join(SAVE_DIR, 'subjects_enhanced.csv'))['PTID'].tolist()
    print(f"Loaded: {len(X)} subjects, shape: {X.shape}")
else:
    features, labels, subjects = [], [], []
    
    for _, row in tqdm(clinical_df.iterrows(), total=len(clinical_df)):
        feat = extractor.extract(row['mri_path'])
        if feat is not None:
            features.append(feat)
            labels.append(row['label_idx'])
            subjects.append(row['PTID'])
    
    X = np.array(features)
    y = np.array(labels)
    
    # Save
    np.save(feature_file, X)
    np.save(label_file, y)
    pd.DataFrame({'PTID': subjects}).to_csv(os.path.join(SAVE_DIR, 'subjects_enhanced.csv'), index=False)
    
    print(f"\nExtracted: {len(X)} subjects")
    print(f"Feature shape: {X.shape}")

print(f"\nClass distribution: {Counter(y)}")

Loading existing features...
Loaded: 196 subjects, shape: (196, 41482)

Class distribution: Counter({np.int64(1): 70, np.int64(0): 67, np.int64(2): 59})


In [7]:
# ========== ADVANCED TRAINING WITH CROSS-VALIDATION ==========
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Standardize features (important for high-dimensional features)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler
joblib.dump(scaler, os.path.join(SAVE_DIR, 'scaler.joblib'))

# Use Stratified K-Fold for robust evaluation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_scores = []
all_preds = []
all_trues = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Compute class weights
    weights = class_weight.compute_sample_weight('balanced', y_train)
    
    # Optimized XGBoost
    clf = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.005,
        max_depth=4,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.6,
        colsample_bylevel=0.8,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective='multi:softprob',
        num_class=3,
        random_state=42,
        early_stopping_rounds=50,
        eval_metric='mlogloss'
    )
    
    clf.fit(
        X_train, y_train,
        sample_weight=weights,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    preds = clf.predict(X_val)
    acc = (preds == y_val).mean()
    fold_scores.append(acc)
    
    all_preds.extend(preds)
    all_trues.extend(y_val)
    
    print(f"Fold {fold+1}: Accuracy = {acc:.4f}")

print(f"\n{'='*50}")
print(f"Cross-Validation Results:")
print(f"Mean Accuracy: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")
print(f"{'='*50}")

# Final report on all folds combined
print("\nCombined Classification Report:")
names = ['CN', 'MCI', 'AD']
print(classification_report(all_trues, all_preds, target_names=names, digits=4))

# Train final model on ALL data for deployment
print("\nTraining final model on all data...")
final_weights = class_weight.compute_sample_weight('balanced', y)

final_clf = XGBClassifier(
    n_estimators=800,
    learning_rate=0.005,
    max_depth=4,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.6,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='multi:softprob',
    num_class=3,
    random_state=42
)

final_clf.fit(X_scaled, y, sample_weight=final_weights)

# Save final model
joblib.dump(final_clf, os.path.join(SAVE_DIR, 'model_enhanced_final.joblib'))
print("Final model saved!")

# Feature importance
importance = pd.DataFrame({
    'feature': range(X.shape[1]),
    'importance': final_clf.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 10 important features:")
print(importance.head(10))

Fold 1: Accuracy = 0.2500
Fold 2: Accuracy = 0.3590
Fold 3: Accuracy = 0.4872
Fold 4: Accuracy = 0.3590
Fold 5: Accuracy = 0.4103

Cross-Validation Results:
Mean Accuracy: 0.3731 (+/- 0.0774)

Combined Classification Report:
              precision    recall  f1-score   support

          CN     0.3333    0.2985    0.3150        67
         MCI     0.3882    0.4714    0.4258        70
          AD     0.3922    0.3390    0.3636        59

    accuracy                         0.3724       196
   macro avg     0.3712    0.3696    0.3681       196
weighted avg     0.3706    0.3724    0.3692       196


Training final model on all data...


KeyboardInterrupt: 

In [8]:
# ========== CLINICAL BIOMARKER FEATURE EXTRACTOR ==========
# Extracts features that actually matter for Alzheimer's:
# - Hippocampal region intensity (proxy for atrophy)
# - Ventricle size (proxy for enlargement)
# - Whole brain volume
# - Intensity histogram statistics

class ClinicalFeatureExtractor:
    def __init__(self):
        self.device = DEVICE
    
    def extract(self, path):
        try:
            img = sitk.GetArrayFromImage(sitk.ReadImage(path))
            img = (img - img.min()) / (img.max() - img.min() + 1e-5)
            
            h, w, d = img.shape
            cx, cy, cz = h//2, w//2, d//2
            
            features = []
            
            # 1. Global intensity statistics
            features.extend([
                img.mean(), img.std(), img.max(), img.min(),
                np.percentile(img, 5), np.percentile(img, 25),
                np.percentile(img, 50), np.percentile(img, 75),
                np.percentile(img, 95),
                np.percentile(img, 95) - np.percentile(img, 5),  # Range
                img.std() / (img.mean() + 1e-5),  # Coefficient of variation
            ])
            
            # 2. Regional features (approximate anatomical locations)
            # Hippocampal region (medial temporal lobe, lower slices)
            hippocampal = img[h//3:h//2, w//3:2*w//3, d//4:d//2]
            features.extend([
                hippocampal.mean(), hippocampal.std(),
                hippocampal.max(), hippocampal.min(),
                np.sum(hippocampal > 0.5) / hippocampal.size,  # Volume fraction
            ])
            
            # Ventricular region (central, CSF = dark)
            ventricular = img[cx-20:cx+20, cy-20:cy+20, cz-15:cz+15]
            features.extend([
                ventricular.mean(), ventricular.std(),
                np.sum(ventricular < 0.2) / ventricular.size,  # CSF fraction
                np.sum(ventricular > 0.6) / ventricular.size,  # White matter fraction
            ])
            
            # 3. Slice-wise gradient features (atrophy = sharper edges)
            for axis in range(3):
                if axis == 0: slices = [img[h//4], img[h//2], img[3*h//4]]
                elif axis == 1: slices = [img[:, w//4], img[:, w//2], img[:, 3*w//4]]
                else: slices = [img[:, :, d//4], img[:, :, d//2], img[:, :, 3*d//4]]
                
                for sl in slices:
                    # Sobel-like edge detection
                    gy, gx = np.gradient(sl)
                    grad_mag = np.sqrt(gx**2 + gy**2)
                    features.extend([
                        grad_mag.mean(), grad_mag.std(), grad_mag.max(),
                        np.sum(grad_mag > 0.1) / grad_mag.size,  # Edge density
                    ])
            
            # 4. Texture features (GLCM-like statistics)
            for axis in range(3):
                if axis == 0: sl = img[h//2]
                elif axis == 1: sl = img[:, w//2]
                else: sl = img[:, :, d//2]
                
                # Co-occurrence statistics
                sl_int = (sl * 7).astype(int)  # 8 gray levels
                features.extend([
                    np.var(sl_int),
                    np.mean(np.abs(np.diff(sl_int, axis=0))),
                    np.mean(np.abs(np.diff(sl_int, axis=1))),
                ])
            
            # 5. Asymmetry features (AD often shows asymmetric atrophy)
            left = img[:, :w//2, :]
            right = img[:, w//2:, :]
            features.extend([
                np.abs(left.mean() - right.mean()),
                np.abs(left.std() - right.std()),
                np.corrcoef(left.flatten()[:10000], right.flatten()[:10000])[0,1] if len(left.flatten()) > 10000 else 0,
            ])
            
            return np.array(features, dtype=np.float32)
            
        except Exception as e:
            print(f"Error: {e}")
            return None

print("Clinical feature extractor ready!")
print("Feature dimensions: ~60 (interpretable biomarkers)")

Clinical feature extractor ready!
Feature dimensions: ~60 (interpretable biomarkers)


In [9]:
# ========== EXTRACT CLINICAL FEATURES ==========
extractor = ClinicalFeatureExtractor()

features, labels, subjects = [], [], []

for _, row in tqdm(clinical_df.iterrows(), total=len(clinical_df)):
    feat = extractor.extract(row['mri_path'])
    if feat is not None:
        features.append(feat)
        labels.append(row['label_idx'])
        subjects.append(row['PTID'])

X_clinical = np.array(features)
y_clinical = np.array(labels)

print(f"\nExtracted: {len(X_clinical)} subjects, shape: {X_clinical.shape}")
print(f"Class distribution: {Counter(y_clinical)}")

# Save
np.save(os.path.join(SAVE_DIR, 'X_clinical.npy'), X_clinical)
np.save(os.path.join(SAVE_DIR, 'y_clinical.npy'), y_clinical)

100%|██████████| 196/196 [04:25<00:00,  1.36s/it]



Extracted: 196 subjects, shape: (196, 68)
Class distribution: Counter({np.int64(1): 70, np.int64(0): 67, np.int64(2): 59})


In [10]:
# ========== TRAIN WITH STRATIFIED K-FOLD ==========
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# Use RobustScaler (better for medical data with outliers)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_clinical)

# Test multiple algorithms
models = {
    'XGBoost': XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, objective='multi:softprob',
        num_class=3, random_state=42
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_split=5,
        class_weight='balanced', random_state=42
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42
    ),
    'SVM': SVC(kernel='rbf', C=1.0, gamma='scale', 
               class_weight='balanced', probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42
    )
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("="*60)
print("MODEL COMPARISON (5-Fold Cross-Validation)")
print("="*60)

best_model = None
best_score = 0

for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y_clinical, cv=skf, scoring='accuracy')
    mean_score = scores.mean()
    std_score = scores.std()
    
    print(f"{name:20s}: {mean_score:.4f} (+/- {std_score:.4f})")
    
    if mean_score > best_score:
        best_score = mean_score
        best_model = (name, model)

print(f"\n{'='*60}")
print(f"BEST MODEL: {best_model[0]} with {best_score:.4f} accuracy")
print(f"{'='*60}")

# Train best model on full data
final_model = best_model[1]
final_model.fit(X_scaled, y_clinical)

# Detailed evaluation
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

y_pred = cross_val_predict(final_model, X_scaled, y_clinical, cv=skf)

print("\nDetailed Classification Report:")
names = ['CN', 'MCI', 'AD']
print(classification_report(y_clinical, y_pred, target_names=names, digits=4))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_clinical, y_pred)
print(f"     CN  MCI  AD")
for i, row in enumerate(cm):
    print(f"{names[i]:4s} {row}")

# Save
joblib.dump(final_model, os.path.join(SAVE_DIR, f'model_{best_model[0].lower()}.joblib'))
joblib.dump(scaler, os.path.join(SAVE_DIR, 'scaler_clinical.joblib'))
print(f"\nModel saved!")

MODEL COMPARISON (5-Fold Cross-Validation)
XGBoost             : 0.4033 (+/- 0.0285)
RandomForest        : 0.3268 (+/- 0.0687)
GradientBoosting    : 0.3931 (+/- 0.0565)
SVM                 : 0.3164 (+/- 0.0389)
LogisticRegression  : 0.3722 (+/- 0.0642)

BEST MODEL: XGBoost with 0.4033 accuracy

Detailed Classification Report:
              precision    recall  f1-score   support

          CN     0.4058    0.4179    0.4118        67
         MCI     0.3929    0.4714    0.4286        70
          AD     0.4186    0.3051    0.3529        59

    accuracy                         0.4031       196
   macro avg     0.4058    0.3981    0.3978       196
weighted avg     0.4050    0.4031    0.4001       196


Confusion Matrix:
     CN  MCI  AD
CN   [28 27 12]
MCI  [24 33 13]
AD   [17 24 18]

Model saved!


In [11]:
import matplotlib.pyplot as plt
import nibabel as nib

# Load first subject from each class
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for i, cat in enumerate(['CN', 'MCI', 'AD']):
    cat_path = os.path.join(NIFTI_BASE, cat)
    files = [f for f in os.listdir(cat_path) if f.endswith('.nii.gz')]
    if not files: 
        print(f"NO FILES in {cat}")
        continue
    
    path = os.path.join(cat_path, files[0])
    
    # Load with nibabel to check orientation
    nii = nib.load(path)
    print(f"\n{cat} - {files[0]}")
    print(f"  Shape: {nii.shape}")
    print(f"  Affine:\n{nii.affine}")
    print(f"  Voxel size: {nii.header.get_zooms()}")
    
    data = nii.get_fdata()
    
    # Show 3 views
    mid = [s//2 for s in data.shape]
    axes[i, 0].imshow(data[mid[0], :, :], cmap='gray')
    axes[i, 0].set_title(f'{cat} - Axial')
    axes[i, 1].imshow(data[:, mid[1], :], cmap='gray')
    axes[i, 1].set_title(f'{cat} - Sagittal')
    axes[i, 2].imshow(data[:, :, mid[2]], cmap='gray')
    axes[i, 2].set_title(f'{cat} - Coronal')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'data_quality_check.png'), dpi=150)
plt.show()


CN - 002_S_0413_2_ADNI_12M4_TS_2.nii.gz
  Shape: (166, 256, 256)
  Affine:
[[   1.19999695    0.            0.          -99.        ]
 [   0.            0.9375        0.          -92.26049805]
 [   0.            0.            0.9375     -106.21150208]
 [   0.            0.            0.            1.        ]]
  Voxel size: (np.float32(1.2), np.float32(0.9375), np.float32(0.9375))

MCI - 002_S_0729_2_ADNI_12M4_TS_2.nii.gz
  Shape: (166, 256, 256)
  Affine:
[[   1.1996994     0.            0.         -100.26799774]
 [   0.            0.9375        0.          -96.48849487]
 [   0.            0.            0.9375     -119.3164978 ]
 [   0.            0.            0.            1.        ]]
  Voxel size: (np.float32(1.1996994), np.float32(0.9375), np.float32(0.9375))

AD - 002_S_0619_2_ADNI_12M4_TS_2.nii.gz
  Shape: (166, 256, 256)
  Affine:
[[   1.20000458    0.            0.         -104.70400238]
 [   0.            0.9375        0.         -105.58050537]
 [   0.            0.        

In [12]:
import SimpleITK as sitk
import numpy as np
from scipy import ndimage
from skimage.measure import regionprops, label

class BrainVolumeExtractor:
    """
    Extracts approximate volumetric features without full FastSurfer.
    Uses intensity thresholding and morphological operations to segment:
    - Brain mask (remove skull)
    - Ventricles (dark CSF)
    - White matter (bright)
    - Gray matter (intermediate)
    """
    
    def __init__(self):
        pass
    
    def _remove_skull(self, img_array):
        """Remove skull using Otsu thresholding + morphological operations."""
        # Normalize
        img = (img_array - img_array.min()) / (img_array.max() - img_array.min())
        
        # Otsu threshold for brain vs background
        from skimage.filters import threshold_otsu
        thresh = threshold_otsu(img)
        
        # Binary mask
        brain_mask = img > thresh * 0.5  # Slightly lower threshold
        
        # Clean up with morphological operations
        brain_mask = ndimage.binary_opening(brain_mask, iterations=2)
        brain_mask = ndimage.binary_closing(brain_mask, iterations=3)
        
        # Keep largest connected component
        labeled = label(brain_mask)
        if labeled.max() > 0:
            largest = np.argmax(np.bincount(labeled.flat)[1:]) + 1
            brain_mask = (labeled == largest)
        
        return brain_mask, img
    
    def _segment_tissues(self, img, brain_mask):
        """Segment CSF, GM, WM using intensity clustering."""
        brain_voxels = img[brain_mask]
        
        if len(brain_voxels) == 0:
            return None, None, None
        
        # K-means-like thresholding (3 classes)
        p10, p40, p70, p90 = np.percentile(brain_voxels, [10, 40, 70, 90])
        
        csf_mask = brain_mask & (img < p10)  # Dark = CSF/ventricles
        gm_mask = brain_mask & (img >= p10) & (img < p70)  # Gray matter
        wm_mask = brain_mask & (img >= p70)  # White matter
        
        return csf_mask, gm_mask, wm_mask
    
    def _extract_hippocampal_region(self, img, brain_mask, shape):
        """Approximate hippocampal region (medial temporal lobe)."""
        h, w, d = shape
        
        # Hippocampus is in medial temporal lobe
        # Approximate location: lower half, medial, posterior
        h_start, h_end = int(h*0.3), int(h*0.6)
        w_start, w_end = int(w*0.3), int(w*0.7)
        d_start, d_end = int(d*0.3), int(d*0.6)
        
        hipp_mask = np.zeros_like(brain_mask)
        hipp_mask[h_start:h_end, w_start:w_end, d_start:d_end] = True
        hipp_mask &= brain_mask
        
        return hipp_mask
    
    def extract(self, path):
        try:
            # Load image
            img_sitk = sitk.ReadImage(path)
            img_array = sitk.GetArrayFromImage(img_sitk)
            
            # Get voxel volume
            spacing = img_sitk.GetSpacing()
            voxel_vol = np.prod(spacing)
            
            # Remove skull
            brain_mask, img_norm = self._remove_skull(img_array)
            
            if brain_mask.sum() < 1000:  # Too small, likely failed
                return None
            
            # Segment tissues
            csf_mask, gm_mask, wm_mask = self._segment_tissues(img_norm, brain_mask)
            
            if csf_mask is None:
                return None
            
            # Approximate hippocampal region
            hipp_mask = self._extract_hippocampal_region(img_norm, brain_mask, img_array.shape)
            
            # Calculate volumes
            features = {
                'brain_vol_mm3': brain_mask.sum() * voxel_vol,
                'csf_vol_mm3': csf_mask.sum() * voxel_vol,
                'gm_vol_mm3': gm_mask.sum() * voxel_vol,
                'wm_vol_mm3': wm_mask.sum() * voxel_vol,
                'hipp_vol_mm3': hipp_mask.sum() * voxel_vol,
                'csf_ratio': csf_mask.sum() / brain_mask.sum(),
                'gm_ratio': gm_mask.sum() / brain_mask.sum(),
                'wm_ratio': wm_mask.sum() / brain_mask.sum(),
                'hipp_ratio': hipp_mask.sum() / brain_mask.sum(),
                'gm_wm_ratio': gm_mask.sum() / (wm_mask.sum() + 1),
                'csf_brain_ratio': csf_mask.sum() / brain_mask.sum(),
            }
            
            # Add shape descriptors
            features['brain_compactness'] = brain_mask.sum() / np.power(brain_mask.sum(), 2/3)
            
            # Regional asymmetry (left-right)
            mid_w = img_array.shape[1] // 2
            left_brain = brain_mask[:, :mid_w, :].sum()
            right_brain = brain_mask[:, mid_w:, :].sum()
            features['brain_asymmetry'] = abs(left_brain - right_brain) / (left_brain + right_brain + 1)
            
            # Hippocampal asymmetry
            left_hipp = hipp_mask[:, :mid_w, :].sum()
            right_hipp = hipp_mask[:, mid_w:, :].sum()
            features['hipp_asymmetry'] = abs(left_hipp - right_hipp) / (left_hipp + right_hipp + 1)
            
            # Ventricular enlargement proxy (CSF in central region)
            cx, cy, cz = [s//2 for s in img_array.shape]
            central_csf = csf_mask[cx-10:cx+10, cy-10:cy+10, cz-10:cz+10].sum()
            features['central_csf_vol'] = central_csf * voxel_vol
            
            return np.array(list(features.values()), dtype=np.float32)
            
        except Exception as e:
            print(f"Error on {os.path.basename(path)}: {e}")
            return None

print("BrainVolumeExtractor ready!")
print("Features: brain_vol, csf_vol, gm_vol, wm_vol, hipp_vol, ratios, asymmetry")

BrainVolumeExtractor ready!
Features: brain_vol, csf_vol, gm_vol, wm_vol, hipp_vol, ratios, asymmetry


In [13]:
# ========== EXTRACT VOLUME-BASED FEATURES ==========
extractor = BrainVolumeExtractor()

features, labels, subjects = [], [], []

for _, row in tqdm(clinical_df.iterrows(), total=len(clinical_df)):
    feat = extractor.extract(row['mri_path'])
    if feat is not None:
        features.append(feat)
        labels.append(row['label_idx'])
        subjects.append(row['PTID'])

X_vol = np.array(features)
y_vol = np.array(labels)

print(f"\nExtracted: {len(X_vol)} subjects, shape: {X_vol.shape}")
print(f"Class distribution: {Counter(y_vol)}")

# Save
np.save(os.path.join(SAVE_DIR, 'X_brain_volumes.npy'), X_vol)
np.save(os.path.join(SAVE_DIR, 'y_brain_volumes.npy'), y_vol)

100%|██████████| 196/196 [05:36<00:00,  1.72s/it]



Extracted: 196 subjects, shape: (196, 15)
Class distribution: Counter({np.int64(1): 70, np.int64(0): 67, np.int64(2): 59})


In [14]:
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Normalize by brain volume (critical!)
brain_volumes = X_vol[:, 0]  # First feature is brain_vol_mm3
X_normalized = X_vol.copy()
for i in range(1, X_vol.shape[1]):
    X_normalized[:, i] = X_vol[:, i] / (brain_volumes + 1)  # Per-brain normalization

# Log transform for volume features
X_log = np.log1p(np.abs(X_normalized))

# Scale
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_log)

# Test models
models = {
    'XGBoost': XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=3,
        subsample=0.8, colsample_bytree=0.8,
        objective='multi:softprob', num_class=3, random_state=42
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_split=5,
        class_weight='balanced', random_state=42
    ),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("="*60)
print("MODEL COMPARISON (Volume-Based Features)")
print("="*60)

best_model = None
best_score = 0

for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y_vol, cv=skf, scoring='accuracy')
    mean = scores.mean()
    std = scores.std()
    print(f"{name:20s}: {mean:.4f} (+/- {std:.4f})")
    
    if mean > best_score:
        best_score = mean
        best_model = model

print(f"\nBest: {best_score:.4f}")

# Train final
best_model.fit(X_scaled, y_vol)
y_pred = cross_val_predict(best_model, X_scaled, y_vol, cv=skf)

print("\nClassification Report:")
print(classification_report(y_vol, y_pred, target_names=['CN', 'MCI', 'AD'], digits=4))

# Feature names for interpretation
feature_names = [
    'brain_vol', 'csf_vol', 'gm_vol', 'wm_vol', 'hipp_vol',
    'csf_ratio', 'gm_ratio', 'wm_ratio', 'hipp_ratio',
    'gm_wm_ratio', 'csf_brain_ratio', 'compactness',
    'brain_asym', 'hipp_asym', 'central_csf'
]

importance = pd.DataFrame({
    'feature': feature_names,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop Features:")
print(importance.head(10))

joblib.dump(best_model, os.path.join(SAVE_DIR, 'model_brain_volumes.joblib'))
joblib.dump(scaler, os.path.join(SAVE_DIR, 'scaler_volumes.joblib'))

MODEL COMPARISON (Volume-Based Features)
XGBoost             : 0.3726 (+/- 0.0455)
RandomForest        : 0.3622 (+/- 0.0517)

Best: 0.3726

Classification Report:
              precision    recall  f1-score   support

          CN     0.3529    0.3582    0.3556        67
         MCI     0.3735    0.4429    0.4052        70
          AD     0.4000    0.3051    0.3462        59

    accuracy                         0.3724       196
   macro avg     0.3755    0.3687    0.3690       196
weighted avg     0.3744    0.3724    0.3705       196


Top Features:
        feature  importance
11  compactness    0.086779
7      wm_ratio    0.080362
8    hipp_ratio    0.074791
14  central_csf    0.070714
13    hipp_asym    0.069968
1       csf_vol    0.069751
4      hipp_vol    0.069476
6      gm_ratio    0.068186
9   gm_wm_ratio    0.062787
12   brain_asym    0.061819


['E:\\ADNI_Local\\outputs\\processed_data\\scaler_volumes.joblib']

In [15]:
# =============================================================================
# ADNI LOCAL PIPEBOOK - Master Notebook for Jupyter
# Copy each CELL block into a separate Jupyter cell and run sequentially
# =============================================================================

# ========== CELL 1: IMPORTS ==========
import os
import sys
import shutil
import subprocess
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import nibabel as nib
import SimpleITK as sitk
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm
from collections import Counter

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight
from sklearn.preprocessing import StandardScaler, RobustScaler
import joblib

print("="*60)
print("ADNI LOCAL PIPELINE - MASTER NOTEBOOK")
print("="*60)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

ADNI LOCAL PIPELINE - MASTER NOTEBOOK
PyTorch: 2.7.1+cpu
CUDA Available: False
Device: CPU


In [16]:

# ========== CELL 2: CONFIGURATION ==========
# EDIT THESE PATHS TO MATCH YOUR SYSTEM
NIFTI_BASE = r"E:\ADNI_NIfTI"           # Your NIfTI files (AD/, MCI/, CN/)
ADNIMERGE_PATH = r"E:\ADNIMERGE.csv"    # ADNIMERGE.csv
FASTSURFER_ROOT = r"E:\ADNI_Local\FastSurfer"  # FastSurfer installation
OUTPUT_ROOT = r"E:\ADNI_Local\outputs"
SAVE_DIR = os.path.join(OUTPUT_ROOT, "processed_data")

# Create directories
for d in [OUTPUT_ROOT, SAVE_DIR]:
    os.makedirs(d, exist_ok=True)

CATEGORIES = ['AD', 'MCI', 'CN']
LABEL_MAP = {'CN': 0, 'MCI': 1, 'AD': 2}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# FLAGS
USE_ALL_SUBJECTS = True      # Set True to include all subjects (matches original Colab behavior)
SKIP_CONVERTERS = False      # Set True to exclude converters (scientifically correct)
REORIENT_NIFTI = True      # Reorient all images to standard RAS space

print(f"\nPaths configured:")
print(f"  NIfTI: {NIFTI_BASE}")
print(f"  ADNIMERGE: {ADNIMERGE_PATH}")
print(f"  FastSurfer: {FASTSURFER_ROOT}")
print(f"  Output: {OUTPUT_ROOT}")


Paths configured:
  NIfTI: E:\ADNI_NIfTI
  ADNIMERGE: E:\ADNIMERGE.csv
  FastSurfer: E:\ADNI_Local\FastSurfer
  Output: E:\ADNI_Local\outputs


In [17]:




# ========== CELL 3: CHECK FOR PREPROCESSED COLAB DATA ==========
# If you saved features from Colab, load them directly and skip to training!
colab_X = os.path.join(SAVE_DIR, 'X_dense_features.npy')
colab_y = os.path.join(SAVE_DIR, 'y_labels.npy')

if os.path.exists(colab_X) and os.path.exists(colab_y):
    print("\n" + "="*60)
    print("FOUND PREPROCESSED COLAB DATA!")
    print("="*60)
    X = np.load(colab_X)
    y = np.load(colab_y)
    print(f"Loaded X: {X.shape}, y: {y.shape}")
    print("Skip to CELL 10 (Train XGBoost)!")
    PRELOADED = True
else:
    print("\nNo preprocessed Colab data found. Will extract features locally.")
    print("Tip: If you have X_dense_features.npy and y_labels.npy from Google Drive,")
    print(f"      copy them to: {SAVE_DIR}")
    PRELOADED = False


No preprocessed Colab data found. Will extract features locally.
Tip: If you have X_dense_features.npy and y_labels.npy from Google Drive,
      copy them to: E:\ADNI_Local\outputs\processed_data


In [18]:




# ========== CELL 3: CHECK FOR PREPROCESSED COLAB DATA ==========
# If you saved features from Colab, load them directly and skip to training!
colab_X = os.path.join(SAVE_DIR, 'X_dense_features.npy')
colab_y = os.path.join(SAVE_DIR, 'y_labels.npy')

if os.path.exists(colab_X) and os.path.exists(colab_y):
    print("\n" + "="*60)
    print("FOUND PREPROCESSED COLAB DATA!")
    print("="*60)
    X = np.load(colab_X)
    y = np.load(colab_y)
    print(f"Loaded X: {X.shape}, y: {y.shape}")
    print("Skip to CELL 10 (Train XGBoost)!")
    PRELOADED = True
else:
    print("\nNo preprocessed Colab data found. Will extract features locally.")
    print("Tip: If you have X_dense_features.npy and y_labels.npy from Google Drive,")
    print(f"      copy them to: {SAVE_DIR}")
    PRELOADED = False


FOUND PREPROCESSED COLAB DATA!
Loaded X: (433, 4096), y: (433,)
Skip to CELL 10 (Train XGBoost)!


In [19]:


# ========== CELL 4: IDENTIFY CONVERTERS ==========
def identify_converters(path):
    """Identify subjects who converted from CN/MCI to AD."""
    if not os.path.exists(path):
        print("WARNING: ADNIMERGE.csv not found. No converters excluded.")
        return set()

    df = pd.read_csv(path, low_memory=False)
    subjects = df.sort_values(['PTID', 'VISCODE'])

    dx_cols = [c for c in ['DX', 'DXCURREN', 'DXCHANGE'] if c in df.columns]
    print(f"Available diagnosis columns: {dx_cols}")

    for col in dx_cols:
        uniques = df[col].dropna().unique()
        print(f"  {col}: {uniques[:10]}")

    def is_converter(group):
        if 'DXCHANGE' in group.columns:
            if any(group['DXCHANGE'].isin([4, 5])):  # 4=MCI->AD, 5=CN->AD
                return True
        if 'DX' in group.columns:
            labels = group['DX'].dropna().unique()
            if 'AD' in labels and len(labels) > 1:
                return True
        return False

    converters = set(subjects.groupby('PTID').filter(is_converter)['PTID'].unique())
    print(f"\nFound {len(converters)} converter subjects.")

    pd.DataFrame({'PTID': list(converters)}).to_csv(
        os.path.join(SAVE_DIR, 'converters.csv'), index=False)

    return converters

converter_ids = identify_converters(ADNIMERGE_PATH)
print(f"Converter IDs (sample): {list(converter_ids)[:5]}")

Available diagnosis columns: ['DX']
  DX: <StringArray>
['CN', 'Dementia', 'MCI']
Length: 3, dtype: str

Found 0 converter subjects.
Converter IDs (sample): []


In [20]:
# ========== CELL 5: BUILD CLINICAL DATAFRAME ==========
data_rows = []

for cat in CATEGORIES:
    cat_path = os.path.join(NIFTI_BASE, cat)
    if not os.path.exists(cat_path):
        print(f"WARNING: {cat_path} not found!")
        continue

    files = [f for f in os.listdir(cat_path) if f.endswith('.nii.gz')]
    print(f"{cat}: {len(files)} NIfTI files")

    for f in files:
        parts = f.split('_')
        # CORRECT PTID extraction: first 3 parts (e.g., 002_S_0413)
        if len(parts) >= 3:
            sid = f"{parts[0]}_{parts[1]}_{parts[2]}"

            # Apply filters
            if USE_ALL_SUBJECTS:
                include = True
            elif SKIP_CONVERTERS and sid in converter_ids:
                include = False
            else:
                include = True

            if include:
                data_rows.append({
                    'PTID': sid,
                    'mri_path': os.path.join(cat_path, f),
                    'label': cat
                })

clinical_df = pd.DataFrame(data_rows).drop_duplicates(subset=['PTID'])
clinical_df['label_idx'] = clinical_df['label'].map(LABEL_MAP)

print(f"\n{'='*60}")
print(f"TOTAL SUBJECTS: {len(clinical_df)}")
print(f"{'='*60}")
print(clinical_df['label'].value_counts())

clinical_df.to_csv(os.path.join(SAVE_DIR, 'clinical_metadata.csv'), index=False)
print(f"\nSaved to: {os.path.join(SAVE_DIR, 'clinical_metadata.csv')}")

AD: 185 NIfTI files
MCI: 529 NIfTI files
CN: 300 NIfTI files

TOTAL SUBJECTS: 283
label
MCI    143
CN      80
AD      60
Name: count, dtype: int64

Saved to: E:\ADNI_Local\outputs\processed_data\clinical_metadata.csv


In [21]:
# ========== CELL 6: DATA QUALITY DIAGNOSTIC ==========
print("\n" + "="*60)
print("DATA QUALITY CHECK")
print("="*60)

# Check if images are registered (consistent affines)
affines = []
shapes = []

for cat in CATEGORIES:
    cat_path = os.path.join(NIFTI_BASE, cat)
    if not os.path.exists(cat_path):
        continue
    files = [f for f in os.listdir(cat_path) if f.endswith('.nii.gz')]
    if files:
        nii = nib.load(os.path.join(cat_path, files[0]))
        affines.append(nii.affine)
        shapes.append(nii.shape)
        print(f"\n{cat} example: {files[0]}")
        print(f"  Shape: {nii.shape}")
        print(f"  Voxel size: {nii.header.get_zooms()}")

# Check consistency
if len(set(tuple(s) for s in shapes)) == 1:
    print("\n✓ All images have consistent dimensions")
else:
    print("\n⚠ WARNING: Images have different dimensions! Registration needed.")

# Quick intensity check per class
print("\n--- Intensity Statistics ---")
for cat in CATEGORIES:
    cat_path = os.path.join(NIFTI_BASE, cat)
    if not os.path.exists(cat_path):
        continue
    files = [f for f in os.listdir(cat_path) if f.endswith('.nii.gz')]
    if files:
        data = nib.load(os.path.join(cat_path, files[0])).get_fdata()
        print(f"{cat}: mean={data.mean():.1f}, std={data.std():.1f}, range=[{data.min():.1f}, {data.max():.1f}]")


DATA QUALITY CHECK

AD example: 002_S_0619_2_ADNI_12M4_TS_2.nii.gz
  Shape: (166, 256, 256)
  Voxel size: (np.float32(1.2), np.float32(0.9375), np.float32(0.9375))

MCI example: 002_S_0729_2_ADNI_12M4_TS_2.nii.gz
  Shape: (166, 256, 256)
  Voxel size: (np.float32(1.1996994), np.float32(0.9375), np.float32(0.9375))

CN example: 002_S_0413_2_ADNI_12M4_TS_2.nii.gz
  Shape: (166, 256, 256)
  Voxel size: (np.float32(1.2), np.float32(0.9375), np.float32(0.9375))

✓ All images have consistent dimensions

--- Intensity Statistics ---
AD: mean=822.8, std=1029.7, range=[0.0, 8667.0]
MCI: mean=977.7, std=1450.3, range=[0.0, 11935.0]
CN: mean=902.9, std=1350.9, range=[0.0, 12645.0]


In [22]:
# ========== CELL 7: ORIGINAL DEEP FEATURE EXTRACTOR (ResNet50) ==========
class DenseVolumeFeatureExtractor:
    """
    EXACT replica from your original Colab code.
    Extracts features from 30 slices (10 per axis) using ResNet50.
    """
    def __init__(self):
        self.device = DEVICE
        self.base_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.encoder = torch.nn.Sequential(*list(self.base_model.children())[:-1])
        self.encoder.eval().to(self.device)

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def extract_features(self, mri_path, slices_per_axis=10):
        try:
            # Load and optionally reorient
            if REORIENT_NIFTI:
                nii = nib.load(mri_path)
                nii = nib.as_closest_canonical(nii)
                img = nii.get_fdata()
            else:
                image = sitk.ReadImage(mri_path)
                img = sitk.GetArrayFromImage(image)

            img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-5)

            all_feats = []
            dims = img.shape

            for axis in range(3):
                start, end = int(dims[axis]*0.3), int(dims[axis]*0.7)
                indices = np.linspace(start, end, slices_per_axis).astype(int)

                for idx in indices:
                    if axis == 0: s = img[idx, :, :]
                    elif axis == 1: s = img[:, idx, :]
                    else: s = img[:, :, idx]

                    pil_img = Image.fromarray((np.stack([s]*3, axis=-1)*255).astype(np.uint8))
                    tensor = self.transform(pil_img).unsqueeze(0).to(self.device)

                    with torch.no_grad():
                        feat = self.encoder(tensor).squeeze().cpu().numpy()
                    all_feats.append(feat)

            mean_feat = np.mean(all_feats, axis=0)
            max_feat = np.max(all_feats, axis=0)
            return np.concatenate([mean_feat, max_feat])
        except Exception as e:
            print(f"  Error extracting features from {os.path.basename(mri_path)}: {e}")
            return None

print("DenseVolumeFeatureExtractor initialized (4096-dim features)")

DenseVolumeFeatureExtractor initialized (4096-dim features)


In [23]:
# ========== CELL 8: EXTRACT DEEP FEATURES ==========
if not PRELOADED:
    extractor = DenseVolumeFeatureExtractor()
    features, labels, valid_subjects = [], [], []

    print(f"\nExtracting features for {len(clinical_df)} subjects...")
    print("This takes ~2-5 minutes on GPU, ~10-15 minutes on CPU")

    for idx, row in tqdm(clinical_df.iterrows(), total=len(clinical_df)):
        feat = extractor.extract_features(row['mri_path'])
        if feat is not None:
            features.append(feat)
            labels.append(row['label_idx'])
            valid_subjects.append(row['PTID'])

    X = np.array(features)
    y = np.array(labels)

    print(f"\n{'='*60}")
    print(f"FEATURE EXTRACTION COMPLETE")
    print(f"{'='*60}")
    print(f"Subjects: {len(X)}")
    print(f"Feature shape: {X.shape}")
    print(f"Class distribution: {dict(Counter(y))}")

    # Save
    np.save(os.path.join(SAVE_DIR, 'X_deep_features.npy'), X)
    np.save(os.path.join(SAVE_DIR, 'y_labels.npy'), y)
    pd.DataFrame({'PTID': valid_subjects}).to_csv(
        os.path.join(SAVE_DIR, 'deep_feature_subjects.csv'), index=False)
    print(f"\nSaved features to {SAVE_DIR}")
else:
    print("Using preloaded features. Skipping extraction.")

Using preloaded features. Skipping extraction.


In [24]:
# ========== CELL 9: TRAIN XGBOOST (DEEP FEATURES) ==========
print("\n" + "="*60)
print("TRAINING XGBOOST ON DEEP FEATURES")
print("="*60)

# Check class distribution
class_counts = Counter(y)
print(f"Class distribution: {dict(class_counts)}")

# Split data
if min(class_counts.values()) >= 2:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    print("Using stratified split")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)
    print("Using random split (insufficient samples for stratification)")

print(f"Train: {len(y_train)}, Test: {len(y_test)}")

# Class weights for imbalance
weights = class_weight.compute_sample_weight(class_weight='balanced', y=y_train)

# EXACT XGBoost parameters from your original Colab code
clf = XGBClassifier(
    n_estimators=500,
    learning_rate=0.01,
    max_depth=6,
    min_child_weight=2,
    subsample=0.7,
    colsample_bytree=0.7,
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    n_jobs=-1
)

print("\nTraining...")
clf.fit(X_train, y_train, sample_weight=weights)

# Evaluate
preds = clf.predict(X_test)
proba = clf.predict_proba(X_test)

print("\n" + "="*60)
print("CLASSIFICATION REPORT - DEEP FEATURES")
print("="*60)
print(classification_report(y_test, preds, target_names=['CN', 'MCI', 'AD'], digits=4))

# Per-class accuracy
from sklearn.metrics import accuracy_score
for i, name in enumerate(['CN', 'MCI', 'AD']):
    mask = y_test == i
    if mask.sum() > 0:
        acc = accuracy_score(y_test[mask], preds[mask])
        print(f"{name} accuracy: {acc:.4f}")

# Save model
model_path = os.path.join(SAVE_DIR, 'mri_classifier_deep_features.joblib')
joblib.dump(clf, model_path)
print(f"\nModel saved: {model_path}")


TRAINING XGBOOST ON DEEP FEATURES
Class distribution: {np.int64(2): 92, np.int64(1): 222, np.int64(0): 119}
Using stratified split
Train: 346, Test: 87

Training...

CLASSIFICATION REPORT - DEEP FEATURES
              precision    recall  f1-score   support

          CN     0.5333    0.3333    0.4103        24
         MCI     0.5735    0.8667    0.6903        45
          AD     0.5000    0.1111    0.1818        18

    accuracy                         0.5632        87
   macro avg     0.5356    0.4370    0.4274        87
weighted avg     0.5472    0.5632    0.5078        87

CN accuracy: 0.3333
MCI accuracy: 0.8667
AD accuracy: 0.1111

Model saved: E:\ADNI_Local\outputs\processed_data\mri_classifier_deep_features.joblib


In [25]:
# ========== CELL 10: FASTSURFER SETUP (VOLUMETRIC FEATURES) ==========
print("\n" + "="*60)
print("FASTSURFER SETUP")
print("="*60)

FS_OUTPUT = os.path.join(OUTPUT_ROOT, "fastsurfer_results")
os.makedirs(FS_OUTPUT, exist_ok=True)

RESULTS_CSV = os.path.join(SAVE_DIR, "brain_volumetric_features.csv")
FAILED_CSV = os.path.join(SAVE_DIR, "failed_subjects.csv")

# FreeSurfer/FastSurfer label IDs for brain regions
REGION_LABELS = {
    'left_hippo': 17, 'right_hippo': 53,
    'left_ventricle': 4, 'right_ventricle': 43,
    'left_entorhinal': 1006, 'right_entorhinal': 2006,
    'left_fusiform': 1007, 'right_fusiform': 2007,
    'left_midtemp': 1015, 'right_midtemp': 2015
}

# Check FastSurfer installation
checkpoint_dir = os.path.join(FASTSURFER_ROOT, "checkpoints")
if os.path.exists(checkpoint_dir):
    checkpoints = os.listdir(checkpoint_dir)
    print(f"FastSurfer checkpoints found: {len(checkpoints)} files")
else:
    print(f"WARNING: No checkpoints found at {checkpoint_dir}")
    print("Run: python FastSurfer\\FastSurferCNN\\download_checkpoints.py --all")

print(f"FastSurfer output: {FS_OUTPUT}")


def run_fastsurfer_windows(subject_id, nifti_path):
    """Run FastSurfer segmentation on Windows."""
    local_input = os.path.join(OUTPUT_ROOT, f"{subject_id}_input.nii.gz")
    if os.path.exists(nifti_path):
        shutil.copy(nifti_path, local_input)

    env = os.environ.copy()
    env["PYTHONPATH"] = f"{FASTSURFER_ROOT};" + env.get("PYTHONPATH", "")

    seg_file = os.path.join(FS_OUTPUT, subject_id, "mri", "aparc.DKTatlas+aseg.deep.mgz")
    conf_file = os.path.join(FS_OUTPUT, subject_id, "mri", "orig.mgz")

    cmd = [
        sys.executable,
        os.path.join(FASTSURFER_ROOT, "FastSurferCNN", "run_prediction.py"),
        "--t1", local_input,
        "--sd", FS_OUTPUT,
        "--sid", subject_id,
        "--device", "cpu",
        "--batch_size", "1",
        "--asegdkt_segfile", seg_file,
        "--conformed_name", conf_file
    ]

    result = subprocess.run(cmd, env=env, capture_output=True, text=True)

    if os.path.exists(local_input):
        os.remove(local_input)

    return os.path.exists(seg_file), result.stderr


def extract_brain_features(subject_id, output_root):
    """Extract volumetric features from FastSurfer segmentation."""
    seg_path = os.path.join(output_root, subject_id, "mri", "aparc.DKTatlas+aseg.deep.mgz")
    if not os.path.exists(seg_path):
        return None

    try:
        img = nib.load(seg_path)
        data = img.get_fdata()
        voxel_vol = np.abs(np.linalg.det(img.affine[:3, :3]))

        stats = {'subject_id': subject_id}

        for name, label_id in REGION_LABELS.items():
            stats[f'{name}_vol_mm3'] = round(np.sum(data == label_id) * voxel_vol, 2)

        stats['total_hippocampus'] = stats['left_hippo_vol_mm3'] + stats['right_hippo_vol_mm3']
        stats['total_ventricles'] = stats['left_ventricle_vol_mm3'] + stats['right_ventricle_vol_mm3']
        stats['whole_brain_vol_mm3'] = round(np.sum(data > 0) * voxel_vol, 2)

        return stats
    except Exception as e:
        print(f"Error extracting features for {subject_id}: {e}")
        return None

print("FastSurfer functions ready!")


FASTSURFER SETUP
FastSurfer checkpoints found: 11 files
FastSurfer output: E:\ADNI_Local\outputs\fastsurfer_results
FastSurfer functions ready!


In [26]:
# ========== CELL 11: FASTSURFER BATCH PROCESSING ==========
print("\n" + "="*60)
print("FASTSURFER BATCH PROCESSING")
print("="*60)

# Check existing progress
processed_ids = set()
if os.path.exists(RESULTS_CSV):
    existing = pd.read_csv(RESULTS_CSV)
    processed_ids = set(existing['subject_id'].tolist())
    print(f"Found {len(processed_ids)} already processed subjects.")

# Initialize failed log
if not os.path.exists(FAILED_CSV):
    with open(FAILED_CSV, "w") as f:
        f.write("subject_id,reason\n")

# Select batch
BATCH_SIZE = int(input("How many subjects to process? (Enter number, or 0 to skip): ") or "0")

if BATCH_SIZE > 0:
    subjects_to_process = [sid for sid in clinical_df['PTID'].unique() 
                            if sid not in processed_ids][:BATCH_SIZE]

    print(f"\nProcessing {len(subjects_to_process)} subjects...")
    print(f"Estimated time: ~{len(subjects_to_process) * 30} minutes on CPU")
    print("Tip: Run this overnight for large batches!\n")

    new_results = []
    for i, sid in enumerate(subjects_to_process):
        row = clinical_df[clinical_df['PTID'] == sid].iloc[0]
        print(f"[{i+1}/{len(subjects_to_process)}] {sid}...", end=" ")

        success, err = run_fastsurfer_windows(sid, row['mri_path'])

        if success:
            stats = extract_brain_features(sid, FS_OUTPUT)
            if stats:
                stats['label'] = row['label']
                new_results.append(stats)
                print(f"OK (Hippo: {stats['total_hippocampus']:.1f})")

                # Save immediately (resumable)
                new_row = pd.DataFrame([stats])
                mode = 'a' if os.path.exists(RESULTS_CSV) else 'w'
                header = not os.path.exists(RESULTS_CSV)
                new_row.to_csv(RESULTS_CSV, mode=mode, header=header, index=False)
            else:
                print("FAIL (feature extraction)")
                with open(FAILED_CSV, "a") as f:
                    f.write(f"{sid},feature_extraction_failed\n")
        else:
            print(f"FAIL (FastSurfer)")
            with open(FAILED_CSV, "a") as f:
                f.write(f"{sid},fastsurfer_failed\n")

    print(f"\nBatch complete. Processed {len(new_results)} new subjects.")
else:
    print("Skipping FastSurfer processing.")


FASTSURFER BATCH PROCESSING

Processing 30 subjects...
Estimated time: ~900 minutes on CPU
Tip: Run this overnight for large batches!

[1/30] 002_S_0619... OK (Hippo: 5888.9)
[2/30] 002_S_0816... OK (Hippo: 6393.2)
[3/30] 002_S_0955... OK (Hippo: 5220.7)
[4/30] 002_S_1018... OK (Hippo: 6536.6)
[5/30] 005_S_0221... OK (Hippo: 5425.9)
[6/30] 005_S_0814... OK (Hippo: 5136.7)
[7/30] 005_S_1341... OK (Hippo: 5716.7)
[8/30] 006_S_0547... OK (Hippo: 6986.5)
[9/30] 006_S_0653... OK (Hippo: 6831.6)
[10/30] 007_S_1248... OK (Hippo: 4609.3)
[11/30] 007_S_1304... OK (Hippo: 5257.0)
[12/30] 007_S_1339... OK (Hippo: 6006.0)
[13/30] 014_S_0328... OK (Hippo: 7667.1)
[14/30] 014_S_0357... OK (Hippo: 6381.7)
[15/30] 021_S_0343... OK (Hippo: 5975.5)
[16/30] 021_S_0753... OK (Hippo: 5632.7)
[17/30] 021_S_1109... OK (Hippo: 5197.6)
[18/30] 024_S_1171... OK (Hippo: 8195.2)
[19/30] 024_S_1307... OK (Hippo: 6761.5)
[20/30] 027_S_0850... OK (Hippo: 5501.7)
[21/30] 027_S_1082... OK (Hippo: 5324.5)
[22/30] 027_

In [27]:





# ========== CELL 12: TRAIN XGBOOST ON VOLUMETRIC FEATURES ==========
print("\n" + "="*60)
print("TRAINING XGBOOST ON VOLUMETRIC FEATURES")
print("="*60)

if not os.path.exists(RESULTS_CSV):
    print("No volumetric data found. Run FastSurfer batch processing first.")
else:
    hippo_df = pd.read_csv(RESULTS_CSV)
    print(f"Loaded {len(hippo_df)} subjects with volumetric features")

    # Merge with labels
    metadata = clinical_df[['PTID', 'label']].drop_duplicates()
    clf_data = hippo_df.merge(metadata, left_on='subject_id', right_on='PTID')

    # Define features
    all_feature_cols = [
        'left_hippo_vol_mm3', 'right_hippo_vol_mm3', 'total_hippocampus',
        'total_ventricles', 'whole_brain_vol_mm3',
        'left_entorhinal_vol_mm3', 'right_entorhinal_vol_mm3',
        'left_fusiform_vol_mm3', 'right_fusiform_vol_mm3',
        'left_midtemp_vol_mm3', 'right_midtemp_vol_mm3'
    ]

    available_features = [f for f in all_feature_cols if f in clf_data.columns]
    print(f"Using features: {available_features}")

    if len(available_features) == 0 or len(clf_data) < 10:
        print("Insufficient data for training.")
    else:
        X_vol = clf_data[available_features].values
        y_vol = clf_data['label'].map(LABEL_MAP).values

        # Normalize by whole brain volume
        brain_vols = clf_data['whole_brain_vol_mm3'].values
        X_vol_norm = X_vol / brain_vols[:, None]

        # Split
        Xv_train, Xv_test, yv_train, yv_test = train_test_split(
            X_vol_norm, y_vol, test_size=0.2, random_state=42, stratify=y_vol)

        # Train
        vol_clf = XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            objective='multi:softprob',
            num_class=3,
            random_state=42
        )
        vol_clf.fit(Xv_train, yv_train)

        # Evaluate
        vol_preds = vol_clf.predict(Xv_test)
        print("\n" + "="*60)
        print("CLASSIFICATION REPORT - VOLUMETRIC FEATURES")
        print("="*60)
        print(classification_report(yv_test, vol_preds, 
                                   target_names=['CN', 'MCI', 'AD'], digits=4))

        # Feature importance
        importance = pd.DataFrame({
            'feature': available_features,
            'importance': vol_clf.feature_importances_
        }).sort_values('importance', ascending=False)
        print("\nTop features:")
        print(importance.head(10))

        # Save
        joblib.dump(vol_clf, os.path.join(SAVE_DIR, 'mri_classifier_volumetric.joblib'))
        print("\nVolumetric model saved!")


TRAINING XGBOOST ON VOLUMETRIC FEATURES
Loaded 30 subjects with volumetric features
Using features: ['left_hippo_vol_mm3', 'right_hippo_vol_mm3', 'total_hippocampus', 'total_ventricles', 'whole_brain_vol_mm3', 'left_entorhinal_vol_mm3', 'right_entorhinal_vol_mm3', 'left_fusiform_vol_mm3', 'right_fusiform_vol_mm3', 'left_midtemp_vol_mm3', 'right_midtemp_vol_mm3']


KeyError: 'label'

In [28]:
# ========== CELL 12: TRAIN XGBOOST ON VOLUMETRIC FEATURES (FIXED) ==========
print("\n" + "="*60)
print("TRAINING XGBOOST ON VOLUMETRIC FEATURES")
print("="*60)

if not os.path.exists(RESULTS_CSV):
    print("No volumetric data found. Run FastSurfer batch processing first.")
else:
    hippo_df = pd.read_csv(RESULTS_CSV)
    print(f"Loaded {len(hippo_df)} subjects with volumetric features")
    print(f"CSV columns: {hippo_df.columns.tolist()}")
    
    # Get labels - handle both cases (label in CSV or needs merge)
    if 'label' not in hippo_df.columns:
        # Merge with clinical_df to get labels
        metadata = clinical_df[['PTID', 'label']].drop_duplicates()
        hippo_df = hippo_df.merge(metadata, left_on='subject_id', right_on='PTID', how='inner')
        print(f"After merge: {len(hippo_df)} subjects matched")
    
    # Define features
    all_feature_cols = [
        'left_hippo_vol_mm3', 'right_hippo_vol_mm3', 'total_hippocampus',
        'total_ventricles', 'whole_brain_vol_mm3',
        'left_entorhinal_vol_mm3', 'right_entorhinal_vol_mm3',
        'left_fusiform_vol_mm3', 'right_fusiform_vol_mm3',
        'left_midtemp_vol_mm3', 'right_midtemp_vol_mm3'
    ]
    
    available_features = [f for f in all_feature_cols if f in hippo_df.columns]
    print(f"Using features: {available_features}")
    
    if len(available_features) == 0 or len(hippo_df) < 10:
        print("Insufficient data for training.")
    else:
        X_vol = hippo_df[available_features].values
        y_vol = hippo_df['label'].map(LABEL_MAP).values
        
        # Normalize by whole brain volume (handle zeros safely)
        brain_vols = hippo_df['whole_brain_vol_mm3'].values
        brain_vols = np.where(brain_vols <= 0, 1, brain_vols)  # Avoid division by zero
        X_vol_norm = X_vol / brain_vols[:, None]
        
        # Check class distribution
        from collections import Counter
        print(f"\nClass distribution: {dict(Counter(y_vol))}")
        
        # Split
        if min(Counter(y_vol).values()) >= 2:
            Xv_train, Xv_test, yv_train, yv_test = train_test_split(
                X_vol_norm, y_vol, test_size=0.2, random_state=42, stratify=y_vol)
        else:
            Xv_train, Xv_test, yv_train, yv_test = train_test_split(
                X_vol_norm, y_vol, test_size=0.2, random_state=42)
        
        print(f"Train: {len(yv_train)}, Test: {len(yv_test)}")
        
        # Train with class weights
        weights = class_weight.compute_sample_weight('balanced', yv_train)
        
        vol_clf = XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            min_child_weight=2,
            subsample=0.8,
            colsample_bytree=0.8,
            objective='multi:softprob',
            num_class=3,
            random_state=42
        )
        
        vol_clf.fit(Xv_train, yv_train, sample_weight=weights)
        
        # Evaluate
        vol_preds = vol_clf.predict(Xv_test)
        print("\n" + "="*60)
        print("CLASSIFICATION REPORT - VOLUMETRIC FEATURES")
        print("="*60)
        print(classification_report(yv_test, vol_preds, 
                                   target_names=['CN', 'MCI', 'AD'], digits=4))
        
        # Confusion matrix
        print("\nConfusion Matrix:")
        cm = confusion_matrix(yv_test, vol_preds)
        print("     CN  MCI  AD")
        for i, name in enumerate(['CN', 'MCI', 'AD']):
            print(f"{name:4s} {cm[i]}")
        
        # Feature importance
        importance = pd.DataFrame({
            'feature': available_features,
            'importance': vol_clf.feature_importances_
        }).sort_values('importance', ascending=False)
        print("\nTop Features:")
        print(importance.head(10))
        
        # Save
        joblib.dump(vol_clf, os.path.join(SAVE_DIR, 'mri_classifier_volumetric.joblib'))
        print("\nVolumetric model saved!")


TRAINING XGBOOST ON VOLUMETRIC FEATURES
Loaded 30 subjects with volumetric features
CSV columns: ['subject_id', 'left_hippo_vol_mm3', 'right_hippo_vol_mm3', 'left_ventricle_vol_mm3', 'right_ventricle_vol_mm3', 'left_entorhinal_vol_mm3', 'right_entorhinal_vol_mm3', 'left_fusiform_vol_mm3', 'right_fusiform_vol_mm3', 'left_midtemp_vol_mm3', 'right_midtemp_vol_mm3', 'total_hippocampus', 'total_ventricles', 'whole_brain_vol_mm3', 'label']
Using features: ['left_hippo_vol_mm3', 'right_hippo_vol_mm3', 'total_hippocampus', 'total_ventricles', 'whole_brain_vol_mm3', 'left_entorhinal_vol_mm3', 'right_entorhinal_vol_mm3', 'left_fusiform_vol_mm3', 'right_fusiform_vol_mm3', 'left_midtemp_vol_mm3', 'right_midtemp_vol_mm3']

Class distribution: {np.int64(2): 30}
Train: 24, Test: 6


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0], got [2]

In [29]:
# ========== CELL 11: FASTSURFER BATCH PROCESSING (ADD MISSING CLASSES) ==========
print("\n" + "="*60)
print("FASTSURFER BATCH PROCESSING - ADDING MISSING CLASSES")
print("="*60)

# Check existing progress
processed_ids = set()
if os.path.exists(RESULTS_CSV):
    existing = pd.read_csv(RESULTS_CSV)
    processed_ids = set(existing['subject_id'].tolist())
    print(f"Found {len(processed_ids)} already processed subjects.")
    print(f"Existing classes: {existing['label'].value_counts().to_dict() if 'label' in existing.columns else 'N/A'}")

# Initialize failed log
if not os.path.exists(FAILED_CSV):
    with open(FAILED_CSV, "w") as f:
        f.write("subject_id,reason\n")

# TARGET: Get 10 CN + 10 MCI (you already have 30 AD)
TARGET_PER_CLASS = {'CN': 10, 'MCI': 10, 'AD': 0}  # AD=0 because you already have enough

subjects_to_process = []
for label, target in TARGET_PER_CLASS.items():
    if target == 0:
        continue
    class_df = clinical_df[
        (clinical_df['label'] == label) & 
        (~clinical_df['PTID'].isin(processed_ids))
    ]
    n = min(target, len(class_df))
    if n > 0:
        sampled = class_df.sample(n, random_state=42)
        subjects_to_process.extend(sampled.to_dict('records'))
        print(f"  Will process {n} new {label} subjects")
    else:
        print(f"  WARNING: No unprocessed {label} subjects available!")

if len(subjects_to_process) == 0:
    print("\nNo new subjects to process. All classes may already be covered.")
else:
    print(f"\nProcessing {len(subjects_to_process)} NEW subjects only...")
    print(f"Estimated time: ~{len(subjects_to_process) * 30} minutes on CPU")

    new_results = []
    for i, row in enumerate(subjects_to_process):
        sid = row['PTID']
        print(f"[{i+1}/{len(subjects_to_process)}] {sid} ({row['label']})... ", end="", flush=True)
        
        success, err = run_fastsurfer_windows(sid, row['mri_path'])
        
        if success:
            stats = extract_brain_features(sid, FS_OUTPUT)
            if stats:
                stats['label'] = row['label']
                new_results.append(stats)
                print(f"OK (Hippo: {stats['total_hippocampus']:.1f})")
                
                # Append to existing CSV
                new_row = pd.DataFrame([stats])
                new_row.to_csv(RESULTS_CSV, mode='a', header=False, index=False)
            else:
                print("FAIL (feature extraction)")
                with open(FAILED_CSV, "a") as f:
                    f.write(f"{sid},feature_extraction_failed\n")
        else:
            print(f"FAIL (FastSurfer)")
            with open(FAILED_CSV, "a") as f:
                f.write(f"{sid},fastsurfer_failed\n")

    print(f"\nDone! Added {len(new_results)} new subjects.")
    print("Run Cell 12 again to train the volumetric model.")


FASTSURFER BATCH PROCESSING - ADDING MISSING CLASSES
Found 30 already processed subjects.
Existing classes: {'AD': 30}
  Will process 10 new CN subjects
  Will process 10 new MCI subjects

Processing 20 NEW subjects only...
Estimated time: ~600 minutes on CPU
[1/20] 031_S_0618 (CN)... OK (Hippo: 8583.0)
[2/20] 002_S_0413 (CN)... OK (Hippo: 7308.7)
[3/20] 021_S_0984 (CN)... OK (Hippo: 8406.2)
[4/20] 033_S_0734 (CN)... OK (Hippo: 7754.4)
[5/20] 014_S_0548 (CN)... OK (Hippo: 7776.7)
[6/20] 029_S_0824 (CN)... OK (Hippo: 6520.9)
[7/20] 006_S_0681 (CN)... OK (Hippo: 7499.8)
[8/20] 133_S_0488 (CN)... OK (Hippo: 8859.4)
[9/20] 005_S_0223 (CN)... OK (Hippo: 6440.2)
[10/20] 007_S_0068 (CN)... OK (Hippo: 7446.3)
[11/20] 130_S_0783 (MCI)... OK (Hippo: 6519.3)
[12/20] 014_S_0557 (MCI)... OK (Hippo: 8219.1)
[13/20] 098_S_0160 (MCI)... OK (Hippo: 5614.6)
[14/20] 127_S_0397 (MCI)... OK (Hippo: 7168.0)
[15/20] 033_S_1309 (MCI)... OK (Hippo: 6050.4)
[16/20] 007_S_0101 (MCI)... OK (Hippo: 7021.9)
[17/20

In [30]:
# ========== CELL 12: TRAIN XGBOOST ON VOLUMETRIC FEATURES (FIXED) ==========
print("\n" + "="*60)
print("TRAINING XGBOOST ON VOLUMETRIC FEATURES")
print("="*60)

if not os.path.exists(RESULTS_CSV):
    print("No volumetric data found. Run FastSurfer batch processing first.")
else:
    hippo_df = pd.read_csv(RESULTS_CSV)
    print(f"Loaded {len(hippo_df)} subjects with volumetric features")
    print(f"CSV columns: {hippo_df.columns.tolist()}")
    
    # Get labels - handle both cases (label in CSV or needs merge)
    if 'label' not in hippo_df.columns:
        # Merge with clinical_df to get labels
        metadata = clinical_df[['PTID', 'label']].drop_duplicates()
        hippo_df = hippo_df.merge(metadata, left_on='subject_id', right_on='PTID', how='inner')
        print(f"After merge: {len(hippo_df)} subjects matched")
    
    # Define features
    all_feature_cols = [
        'left_hippo_vol_mm3', 'right_hippo_vol_mm3', 'total_hippocampus',
        'total_ventricles', 'whole_brain_vol_mm3',
        'left_entorhinal_vol_mm3', 'right_entorhinal_vol_mm3',
        'left_fusiform_vol_mm3', 'right_fusiform_vol_mm3',
        'left_midtemp_vol_mm3', 'right_midtemp_vol_mm3'
    ]
    
    available_features = [f for f in all_feature_cols if f in hippo_df.columns]
    print(f"Using features: {available_features}")
    
    if len(available_features) == 0 or len(hippo_df) < 10:
        print("Insufficient data for training.")
    else:
        X_vol = hippo_df[available_features].values
        y_vol = hippo_df['label'].map(LABEL_MAP).values
        
        # Normalize by whole brain volume (handle zeros safely)
        brain_vols = hippo_df['whole_brain_vol_mm3'].values
        brain_vols = np.where(brain_vols <= 0, 1, brain_vols)  # Avoid division by zero
        X_vol_norm = X_vol / brain_vols[:, None]
        
        # Check class distribution
        from collections import Counter
        print(f"\nClass distribution: {dict(Counter(y_vol))}")
        
        # Split
        if min(Counter(y_vol).values()) >= 2:
            Xv_train, Xv_test, yv_train, yv_test = train_test_split(
                X_vol_norm, y_vol, test_size=0.2, random_state=42, stratify=y_vol)
        else:
            Xv_train, Xv_test, yv_train, yv_test = train_test_split(
                X_vol_norm, y_vol, test_size=0.2, random_state=42)
        
        print(f"Train: {len(yv_train)}, Test: {len(yv_test)}")
        
        # Train with class weights
        weights = class_weight.compute_sample_weight('balanced', yv_train)
        
        vol_clf = XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            min_child_weight=2,
            subsample=0.8,
            colsample_bytree=0.8,
            objective='multi:softprob',
            num_class=3,
            random_state=42
        )
        
        vol_clf.fit(Xv_train, yv_train, sample_weight=weights)
        
        # Evaluate
        vol_preds = vol_clf.predict(Xv_test)
        print("\n" + "="*60)
        print("CLASSIFICATION REPORT - VOLUMETRIC FEATURES")
        print("="*60)
        print(classification_report(yv_test, vol_preds, 
                                   target_names=['CN', 'MCI', 'AD'], digits=4))
        
        # Confusion matrix
        print("\nConfusion Matrix:")
        cm = confusion_matrix(yv_test, vol_preds)
        print("     CN  MCI  AD")
        for i, name in enumerate(['CN', 'MCI', 'AD']):
            print(f"{name:4s} {cm[i]}")
        
        # Feature importance
        importance = pd.DataFrame({
            'feature': available_features,
            'importance': vol_clf.feature_importances_
        }).sort_values('importance', ascending=False)
        print("\nTop Features:")
        print(importance.head(10))
        
        # Save
        joblib.dump(vol_clf, os.path.join(SAVE_DIR, 'mri_classifier_volumetric.joblib'))
        print("\nVolumetric model saved!")


TRAINING XGBOOST ON VOLUMETRIC FEATURES
Loaded 50 subjects with volumetric features
CSV columns: ['subject_id', 'left_hippo_vol_mm3', 'right_hippo_vol_mm3', 'left_ventricle_vol_mm3', 'right_ventricle_vol_mm3', 'left_entorhinal_vol_mm3', 'right_entorhinal_vol_mm3', 'left_fusiform_vol_mm3', 'right_fusiform_vol_mm3', 'left_midtemp_vol_mm3', 'right_midtemp_vol_mm3', 'total_hippocampus', 'total_ventricles', 'whole_brain_vol_mm3', 'label']
Using features: ['left_hippo_vol_mm3', 'right_hippo_vol_mm3', 'total_hippocampus', 'total_ventricles', 'whole_brain_vol_mm3', 'left_entorhinal_vol_mm3', 'right_entorhinal_vol_mm3', 'left_fusiform_vol_mm3', 'right_fusiform_vol_mm3', 'left_midtemp_vol_mm3', 'right_midtemp_vol_mm3']

Class distribution: {np.int64(2): 30, np.int64(0): 10, np.int64(1): 10}
Train: 40, Test: 10

CLASSIFICATION REPORT - VOLUMETRIC FEATURES
              precision    recall  f1-score   support

          CN     1.0000    0.5000    0.6667         2
         MCI     0.0000    0.0000

In [31]:
# ========== CELL 13: SUMMARY ==========
print("\n" + "="*60)
print("PIPELINE COMPLETE")
print("="*60)
print(f"All outputs saved to: {SAVE_DIR}")
print("\nFiles generated:")
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024
    print(f"  {f} ({size:.1f} KB)")

print("\n" + "="*60)
print("NEXT STEPS:")
print("="*60)
print("1. If deep feature accuracy is < 50%: Your NIfTI files may need")
print("   registration/MNI alignment. Consider re-downloading from Colab.")
print("2. For best accuracy: Process all subjects with FastSurfer")
print("   (run CELL 11 overnight with BATCH_SIZE = all subjects)")
print("3. The volumetric model typically achieves 65-75% accuracy")
print("   with sufficient FastSurfer-processed subjects.")
print("="*60)


PIPELINE COMPLETE
All outputs saved to: E:\ADNI_Local\outputs\processed_data

Files generated:
  brain_volumetric_features.csv (6.6 KB)
  clinical_metadata.csv (19.2 KB)
  converters.csv (0.0 KB)
  data_quality_check.png (1157.3 KB)
  deep_feature_subjects.csv (2.3 KB)
  deep_model_predictions.csv (1.6 KB)
  failed_subjects.csv (0.0 KB)
  model_brain_volumes.joblib (949.7 KB)
  model_xgboost.joblib (1642.7 KB)
  mri_classifier_deep_features.joblib (2142.3 KB)
  mri_classifier_volumetric.joblib (703.7 KB)
  scaler.joblib (972.8 KB)
  scaler_clinical.joblib (1.3 KB)
  scaler_volumes.joblib (0.7 KB)
  subjects_enhanced.csv (2.3 KB)
  X_brain_volumes.npy (11.6 KB)
  X_clinical.npy (52.2 KB)
  X_deep_features.npy (3136.1 KB)
  X_dense_features.npy (6928.1 KB)
  X_enhanced.npy (63519.4 KB)
  y_brain_volumes.npy (1.7 KB)
  y_clinical.npy (1.7 KB)
  y_deep_labels.npy (1.7 KB)
  y_enhanced.npy (1.7 KB)
  y_labels.npy (3.5 KB)

NEXT STEPS:
1. If deep feature accuracy is < 50%: Your NIfTI files 

In [32]:
# ========== SAVE SESSION STATE ==========
import pickle
import joblib

# Save all important variables
session = {
    'clinical_df': clinical_df,
    'converter_ids': converter_ids,
    'X': X if 'X' in globals() else None,
    'y': y if 'y' in globals() else None,
    'clf': clf if 'clf' in globals() else None,
    'vol_clf': vol_clf if 'vol_clf' in globals() else None,
    'processed_ids': processed_ids,
    'hippo_df': pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else None
}

# Save to disk
session_path = os.path.join(SAVE_DIR, 'session_state.pkl')
with open(session_path, 'wb') as f:
    pickle.dump(session, f)

print(f"Session saved to: {session_path}")
print(f"Size: {os.path.getsize(session_path)/1024/1024:.1f} MB")

# Also save a summary
summary = f"""
ADNI Pipeline Session Summary
============================
Date: {pd.Timestamp.now()}
Subjects processed: {len(processed_ids)}
Volumetric subjects: {len(session['hippo_df']) if session['hippo_df'] is not None else 0}
Deep feature model: {'Trained' if session['clf'] is not None else 'Not trained'}
Volumetric model: {'Trained' if session['vol_clf'] is not None else 'Not trained'}

Files in {SAVE_DIR}:
"""
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024
    summary += f"  {f:40s} {size:8.1f} KB\n"

with open(os.path.join(SAVE_DIR, 'session_summary.txt'), 'w') as f:
    f.write(summary)

print(summary)

Session saved to: E:\ADNI_Local\outputs\processed_data\session_state.pkl
Size: 9.6 MB

ADNI Pipeline Session Summary
Date: 2026-06-17 10:48:56.363707
Subjects processed: 30
Volumetric subjects: 50
Deep feature model: Trained
Volumetric model: Trained

Files in E:\ADNI_Local\outputs\processed_data:
  X_brain_volumes.npy                          11.6 KB
  X_clinical.npy                               52.2 KB
  X_deep_features.npy                        3136.1 KB
  X_dense_features.npy                       6928.1 KB
  X_enhanced.npy                            63519.4 KB
  brain_volumetric_features.csv                 6.6 KB
  clinical_metadata.csv                        19.2 KB
  converters.csv                                0.0 KB
  data_quality_check.png                     1157.3 KB
  deep_feature_subjects.csv                     2.3 KB
  deep_model_predictions.csv                    1.6 KB
  failed_subjects.csv                           0.0 KB
  model_brain_volumes.joblib             